### LLM Chain 만들기


## 1. 환경 구성

### 1) 라이브러리 설치

In [ ]:
# uv add python-dotenv langchain langchain-openai

### 2) GROQ 인증키 설정

In [2]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:3])

gsk


In [3]:
import langchain
print(langchain.__version__)

1.4.3


In [4]:
from langchain_openai import ChatOpenAI

# Groq(OpenAI 호환) 공통 설정 — 모델/온도 변경은 이곳만 수정
GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트
GROQ_MODEL = "openai/gpt-oss-120b"  # 대안: "qwen/qwen3.8-27b"
#MOVIE_MODEL = "qwen/qwen3.8-27b"  # 영화 추천 예제용 
TEMPERATURE = 0.7


def get_llm(model: str = GROQ_MODEL, temperature: float = TEMPERATURE) -> ChatOpenAI:
    """Groq 엔드포인트를 사용하는 ChatOpenAI 인스턴스를 생성합니다."""
    return ChatOpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
        model=model,
        temperature=temperature,
    )


# 기본 llm (이후 셀에서 재사용)
llm = get_llm()

## 2. LLM Chain

### 1) Prompt + LLM


In [5]:
# model: 공통 설정 셀의 llm 사용

# chain 실행
result = llm.invoke("인공지능 모델의 학습 원리에 대하여 쉽게 설명해 주세요.")
print(type(result))
print(result)

<class 'langchain_core.messages.ai.AIMessage'>
content='## 인공지능 모델이 어떻게 배우는지 쉽게 풀어볼게요  \n\n### 1. “학생”과 “문제집”을 생각해 보세요\n- **학생** : 우리가 만들고 싶은 인공지능 모델 (예: 사진을 보고 고양이·강아지를 구분하는 모델)  \n- **문제집** : 모델에게 보여줄 **훈련 데이터** (입력과 정답이 함께 적힌 예시들)  \n\n학생이 문제집을 풀면서 점점 실력이 늘어나듯이, 모델도 데이터를 보면서 스스로 규칙을 찾아갑니다.\n\n---\n\n## 2. 학습 과정의 핵심 단계\n\n| 단계 | 무슨 일? | 비유 |\n|------|----------|------|\n| **① 입력** | 사진, 텍스트, 소리 등 데이터를 모델에 넣는다. | 문제집에 문제를 꺼내는 것 |\n| **② 예측** | 현재 모델이 만든 ‘답’(예: “고양이일 확률 0.6”)을 출력한다. | 학생이 문제를 풀고 답을 적는 것 |\n| **③ 정답 확인** | 데이터에 포함된 정답(라벨)과 비교한다. | 정답지와 채점한다 |\n| **④ 손실(오차) 계산** | 예측과 정답 사이의 차이를 수치화한다. | “틀린 점수”를 계산 |\n| **⑤ 피드백(가중치 조정)** | 손실을 줄이도록 모델 내부 파라미터(가중치)를 조금씩 바꾼다. | “틀렸으니 다음엔 이렇게 생각해 보자” 라고 스스로 교정 |\n| **⑥ 반복** | 위 과정을 수천·수만 번 반복한다. | 문제집을 여러 번 풀면서 실력 향상 |\n\n---\n\n## 3. ‘가중치’가 뭔가요?\n\n- **가중치**는 모델이 입력을 어떻게 변환해서 결과를 내놓을지 결정하는 숫자들의 집합입니다.  \n- 마치 **학생의 사고 방식**처럼, “눈을 보면 ‘털이 많다’ → ‘고양이일 확률을 올린다’” 같은 규칙을 담고 있습니다.  \n- 학습이 진행될수록 가중치가 점점 “정답에 가까운” 사고 방식을 배우게 됩니다.\n\n---\n\n## 4. 

In [6]:
print(result.content)

## 인공지능 모델이 어떻게 배우는지 쉽게 풀어볼게요  

### 1. “학생”과 “문제집”을 생각해 보세요
- **학생** : 우리가 만들고 싶은 인공지능 모델 (예: 사진을 보고 고양이·강아지를 구분하는 모델)  
- **문제집** : 모델에게 보여줄 **훈련 데이터** (입력과 정답이 함께 적힌 예시들)  

학생이 문제집을 풀면서 점점 실력이 늘어나듯이, 모델도 데이터를 보면서 스스로 규칙을 찾아갑니다.

---

## 2. 학습 과정의 핵심 단계

| 단계 | 무슨 일? | 비유 |
|------|----------|------|
| **① 입력** | 사진, 텍스트, 소리 등 데이터를 모델에 넣는다. | 문제집에 문제를 꺼내는 것 |
| **② 예측** | 현재 모델이 만든 ‘답’(예: “고양이일 확률 0.6”)을 출력한다. | 학생이 문제를 풀고 답을 적는 것 |
| **③ 정답 확인** | 데이터에 포함된 정답(라벨)과 비교한다. | 정답지와 채점한다 |
| **④ 손실(오차) 계산** | 예측과 정답 사이의 차이를 수치화한다. | “틀린 점수”를 계산 |
| **⑤ 피드백(가중치 조정)** | 손실을 줄이도록 모델 내부 파라미터(가중치)를 조금씩 바꾼다. | “틀렸으니 다음엔 이렇게 생각해 보자” 라고 스스로 교정 |
| **⑥ 반복** | 위 과정을 수천·수만 번 반복한다. | 문제집을 여러 번 풀면서 실력 향상 |

---

## 3. ‘가중치’가 뭔가요?

- **가중치**는 모델이 입력을 어떻게 변환해서 결과를 내놓을지 결정하는 숫자들의 집합입니다.  
- 마치 **학생의 사고 방식**처럼, “눈을 보면 ‘털이 많다’ → ‘고양이일 확률을 올린다’” 같은 규칙을 담고 있습니다.  
- 학습이 진행될수록 가중치가 점점 “정답에 가까운” 사고 방식을 배우게 됩니다.

---

## 4. 어떻게 ‘조정’하나요? → **경사 하강법(Gradient Descent)**

1. **오차가 큰 방향**을 찾는다. → “여기서 많이 틀렸어!”  
2.

### 2) PromptTemplate + LLM

In [9]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("You are an expert in AI Expert. Answer the question.\
                                       <Question>: {input}에 대해 쉽게 설명해주세요.")

print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.prompt.PromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} template='You are an expert in AI Expert. Answer the question.                                       <Question>: {input}에 대해 쉽게 설명해주세요.'


In [10]:
# llm: 공통 설정 셀의 llm 사용
# chain 연결 (LCEL)
chain = prompt | llm
print(type(chain))

# chain 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.ai.AIMessage'>
content='### 인공지능 모델이 “학습”하는 원리 (쉽게 풀어 설명)\n\n#### 1. **학습이란 무엇인가?**\n인공지능, 특히 **머신러닝 모델**은 “데이터를 보고” 스스로 규칙을 찾아내는 과정을 **학습**이라고 부릅니다.  \n마치 아이가 여러 번 반복해서 그림을 그리며 “어떤 색을 쓰면 파란 하늘이 되는지” 배우는 것과 비슷합니다.\n\n---\n\n#### 2. **학습에 필요한 세 가지 기본 요소**\n\n| 요소 | 역할 | 쉬운 비유 |\n|------|------|-----------|\n| **데이터** | 모델이 보고 배울 원본 | 교과서·연습문제 |\n| **모델(함수)** | 입력을 받아서 예측값을 내는 수식 | 학생이 문제를 풀 때 쓰는 ‘공식’ |\n| **목표(손실·Loss)** | “얼마나 틀렸는가”를 알려주는 점수 | 시험 채점표 (맞으면 0점, 틀리면 점수 부여) |\n\n---\n\n#### 3. **학습 흐름을 단계별로 살펴보면**\n\n1. **입력‑출력 쌍을 준비**  \n   - **입력**: 사진, 텍스트, 센서값 등  \n   - **정답(라벨)**: 사진이면 “고양이”, “강아지” 등  \n\n2. **예측값 만들기**  \n   - 모델에 입력을 넣으면 **예측값**이 나옵니다.  \n   - 예: 사진을 넣었을 때 “고양이 0.7, 강아지 0.3” 같은 확률값.\n\n3. **오차(손실) 계산**  \n   - 예측값과 정답을 비교해 **손실**을 구합니다.  \n   - 손실이 클수록 “틀렸음”을 의미합니다. (예: 정답이 ‘고양이’인데 0.3만 나오면 손실이 큼)\n\n4. **모델을 조금씩 고치기 (최적화)**  \n   - **경사 하강법(Gradient Descent)**이라는 방법으로 손실을 줄이는 방

In [11]:
print(result.content)

### 인공지능 모델이 “학습”하는 원리 (쉽게 풀어 설명)

#### 1. **학습이란 무엇인가?**
인공지능, 특히 **머신러닝 모델**은 “데이터를 보고” 스스로 규칙을 찾아내는 과정을 **학습**이라고 부릅니다.  
마치 아이가 여러 번 반복해서 그림을 그리며 “어떤 색을 쓰면 파란 하늘이 되는지” 배우는 것과 비슷합니다.

---

#### 2. **학습에 필요한 세 가지 기본 요소**

| 요소 | 역할 | 쉬운 비유 |
|------|------|-----------|
| **데이터** | 모델이 보고 배울 원본 | 교과서·연습문제 |
| **모델(함수)** | 입력을 받아서 예측값을 내는 수식 | 학생이 문제를 풀 때 쓰는 ‘공식’ |
| **목표(손실·Loss)** | “얼마나 틀렸는가”를 알려주는 점수 | 시험 채점표 (맞으면 0점, 틀리면 점수 부여) |

---

#### 3. **학습 흐름을 단계별로 살펴보면**

1. **입력‑출력 쌍을 준비**  
   - **입력**: 사진, 텍스트, 센서값 등  
   - **정답(라벨)**: 사진이면 “고양이”, “강아지” 등  

2. **예측값 만들기**  
   - 모델에 입력을 넣으면 **예측값**이 나옵니다.  
   - 예: 사진을 넣었을 때 “고양이 0.7, 강아지 0.3” 같은 확률값.

3. **오차(손실) 계산**  
   - 예측값과 정답을 비교해 **손실**을 구합니다.  
   - 손실이 클수록 “틀렸음”을 의미합니다. (예: 정답이 ‘고양이’인데 0.3만 나오면 손실이 큼)

4. **모델을 조금씩 고치기 (최적화)**  
   - **경사 하강법(Gradient Descent)**이라는 방법으로 손실을 줄이는 방향으로 **파라미터(가중치)**를 조정합니다.  
   - 마치 “시험 점수가 낮으면 공부 방법을 바꾸는” 과정과 비슷합니다.  
   - 이 과정을 **반복**(에폭)하면서 손실이 점점 작아집니다.

5. **학습 종료**  
   - 손실이 충분히 낮아지거나

### 3) PromptTemplate + LLM(invoke()) + StrOutputParser

In [12]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

output_parser = StrOutputParser()

# 2. chain 생성 (LCEL)
chain = prompt | llm | output_parser
print(type(chain))

# 3. chain의 invoke 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.base.TextAccessor'>
### 인공지능 모델이 어떻게 “학습”하나요?  
(쉽게 이해할 수 있도록 비유와 함께 설명합니다)

---

## 1. 기본 아이디어: **예시를 보고 배운다**  

- **학생이 교과서와 문제를 풀면서 공부**하는 것과 비슷합니다.  
- 인공지능(AI) 모델은 **많은 예시(데이터)** 를 보고 “정답”을 맞추는 방법을 스스로 찾아냅니다.

---

## 2. 구성 요소

| 용어 | 쉬운 설명 | 역할 |
|------|-----------|------|
| **데이터** | 모델에게 보여줄 사진, 글, 소리 등 | 학습의 재료 |
| **입력 (Input)** | 데이터 중 모델이 보는 부분 (예: 사진의 픽셀) | 모델이 받아들이는 정보 |
| **출력 (Output)** | 모델이 만든 답 (예: “고양이” 혹은 “강아지”) | 예측 결과 |
| **정답 라벨 (Label)** | 입력에 맞는 정답 (예: 사진에 실제로 고양이가 있으면 “고양이”) | 모델이 목표로 삼는 값 |
| **손실 함수 (Loss)** | 정답과 모델이 낸 답 사이 차이를 수치화 | “얼마나 틀렸는가”를 알려줌 |
| **옵티마이저 / 경사 하강법** | 손실을 줄이기 위해 모델을 조금씩 바꾸는 방법 | 모델을 개선하는 “학습 도구” |

---

## 3. 학습 흐름 (한 번의 반복, “에포크”)

1. **입력 데이터를 모델에 넣는다**  
   - 예: 28×28 픽셀의 손글씨 사진을 넣음.

2. **모델이 예측을 만든다**  
   - 현재 모델 파라미터(가중치)로 “이 사진은 3번 숫자일까?” 라는 확률을 계산.

3. **손실을 계산한다**  
   - 예측이 “3”이라고 했지만 실제 정답이 “5”라면, 손실 함수가 큰 값을 반환.

4. **손실을 줄이는 방향을 찾

### 4) PromptTemplate + LLM(stream()) + StrOutputParser

In [13]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

# chain 연결 (LCEL)
chain = prompt | llm | StrOutputParser()

# 스트리밍 출력을 위한 요청
answer = chain.stream({"input": "인공지능 모델의 학습 원리"})
# 스트리밍 출력
#print(answer)

for token in answer:
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

## 인공지능 모델이 어떻게 배우는지 쉽게 설명해볼게요  

### 1. **학습은 ‘경험을 통해 규칙을 찾는 과정’**
우리 사람도 새로운 일을 할 때 “시도‑실패‑수정”을 반복하면서 배워요.  
예를 들어, 자전거를 탈 때 처음엔 넘어지지만, 몇 번을 타면서 균형을 잡는 방법을 몸에 익히죠.  
인공지능(AI) 모델도 비슷하게 **데이터(경험)** 를 가지고 **반복적으로 시도하고** **오차를 줄여가며** 스스로 규칙을 찾아갑니다.

---

### 2. **핵심 요소 3가지**

| 요소 | 설명 | 비유 |
|------|------|------|
| **데이터** | 모델이 배우는 재료. 사진, 텍스트, 소리 등 | 책, 영상, 연습문제 |
| **모델(구조)** | 데이터를 처리하고 예측을 만드는 ‘뇌’와 비슷한 구조. 주로 **신경망**을 사용 | 뇌의 신경세포(뉴런)와 연결(시냅스) |
| **손실 함수(오차 측정)** | 모델이 얼마나 틀렸는지 수치화. 이 값을 최소화하도록 학습 | 시험 점수(높을수록 좋음) |

---

### 3. **학습 과정 (간단히 5단계)**

1. **초기화**  
   - 모델 안에 있는 가중치(연결 강도)를 무작위로 설정합니다.  
   - 마치 처음에 아무것도 모르는 상태와 비슷해요.

2. **입력 → 예측**  
   - 데이터(예: 고양이 사진)를 모델에 넣으면, 현재 가중치에 따라 **예측값**(고양이인지 아닌지)을 내놓습니다.

3. **오차 계산**  
   - 예측값과 정답(라벨, 여기서는 “고양이”)을 비교해 **손실 함수**로 오차를 구합니다.  
   - 오차가 클수록 “틀렸어요!” 라는 신호.

4. **가중치 업데이트**  
   - 오차를 줄이기 위해 **경사 하강법** 같은 최적화 알고리즘을 사용해 가중치를 조금씩 조정합니다.  
   - “다음엔 조금 더 정확히 맞춰볼게!” 라는 의미.

5. **반복 (Epoch)**  
   - 위 과정을 전체 데이터에 대해 여러 번(수십, 수

##### 2) Multiple Chains
* Multi Chain을 활용한 영화 추천 및 줄거리 요약 

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고,\
                                            줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요")

# 영화 예제용 모델 (공통 설정 셀의 get_llm 사용)
llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
movie = chain1.invoke({"genre": "Drama"})  # 영화 제목 얻기

print(type(movie))
print(" ===> 추천된 영화:")  # movie 값 출력
print(movie)


<class 'langchain_core.messages.base.TextAccessor'>
 ===> 추천된 영화:
**추천 영화: 《소울 (Soul)》 (2020)**  

- **장르**: 드라마·애니메이션·음악  
- **감독**: 피트 닥터톤, 케프 파워스  
- **주연**: 제이미 폭스 (목소리), 티나 페이 (목소리)  

### 간단 소개
피아노 연주가 꿈인 중학교 음악 교사 ‘조 가드너(제이미 폭스)’는 어느 날 교통사고를 당하고, 영혼이 ‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘‘

In [15]:
# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)
print(chain2)

# 실행: "SF" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:\n", response) 

first={
  movie: ChatPromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, template='{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.'), additional_kwargs={})])
         | ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001ABC6D57920>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001ABDD2BBF50>, root_client=<openai.OpenAI object at 0x000001ABC7088170>, root_async_client=<openai.AsyncOpenAI object at 0x000001ABC6C33F20>, model_name='openai/gpt-oss-120b', temperature=0.7, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.groq.com/openai/v1', stream_chunk_timeout=120.0)
         | StrOutputParser()
} middle=[ChatPromptTemplate

##### chain1과 chain2에서 영화 제목이 일관되게 전달 되도록 변경 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from pprint import pprint

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고, 줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요.")

llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: "Drama" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:")
pprint(response)

In [ ]:
# 모델 설정 (공통 설정 셀의 get_llm 사용)
model = get_llm(GROQ_MODEL)

# 멀티 체인 실행 함수
def movie_chain(genre):
    print(f"\n🔹 1단계: '{genre}' 장르 영화 추천")
    
    # 체인 1: 영화 추천
    response1 = model.invoke(f"{genre} 장르에서 유명한 영화 제목 하나만 알려줘")
    movie = response1.content.strip()
    print(f"   → 추천 영화: {movie}")
    
    print(f"\n 2단계: '{movie}' 영화 정보 조회")
    
    # 체인 2: 영화 정보
    response2 = model.invoke(f"{movie} 영화에 대해 간단히 알려줘. 줄거리, 감독, 출연진 포함해서")
    info = response2.content
    print(f"   → 영화 정보:\n{info}")
    
    return movie, info

# 실행
genre = input("원하는 장르를 입력하세요: ")
movie, info = movie_chain(genre)
print(f"\n 최종 결과: '{movie}' 영화가 추천되었습니다!")